# Winsorizer: cap statistical outliers

This notebook follows the Feature-engine outliers example. `Winsorizer` finds unusually small or large values using a statistical rule, then replaces them with the selected boundary. It keeps every row, unlike an outlier trimmer.

The example uses `houseprice.csv`. We inspect `LotArea`, `GrLivArea`, `TotalBsmtSF`, and `SalePrice`, fit the transformer on training data only, compare before and after distributions, and inspect optional outlier indicators.

## Why cap outliers?

Extreme values can pull means and regression lines, inflate variance, and affect distance-based models. Capping limits their influence while preserving the observation. It should be used only when the extreme values are errors, rare cases that should have limited influence, or values for which a ceiling/floor is justified.

In [ ]:
import importlib.util, subprocess, sys
packages = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "seaborn": "seaborn", "sklearn": "scikit-learn", "feature_engine": "feature-engine"}
missing = [pkg for module, pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from feature_engine.outliers import Winsorizer
sns.set_theme(style="whitegrid", context="notebook")

## 1. Load and inspect the housing data

The file is stored in the project's `imputation` folder. The path search makes the notebook work from either the project root or this folder.

In [ ]:
paths = [Path("houseprice.csv"), Path("../imputation/houseprice.csv"), Path("feature-engineering/imputation/houseprice.csv")]
path = next((p for p in paths if p.exists()), None)
if path is None: raise FileNotFoundError("Could not find houseprice.csv")
data = pd.read_csv(path)
features = ["LotArea", "GrLivArea", "TotalBsmtSF", "SalePrice"]
data[features] = data[features].apply(pd.to_numeric, errors="coerce")
data = data.dropna(subset=features).reset_index(drop=True)
display(data[features].describe().round(1))
display(data[features].isna().sum().rename("missing").to_frame())

## 2. Visualise possible outliers

Histograms show skew and boxplots make points beyond the whiskers easy to see. A boxplot flag is a visual convention, not proof that a value is wrong.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for col, hist_ax, box_ax in zip(features, axes[0], axes[1]):
    sns.histplot(data[col], kde=True, ax=hist_ax, color="#4C78A8")
    hist_ax.set_title(f"Raw distribution: {col}")
    sns.boxplot(x=data[col], ax=box_ax, color="#F58518")
    box_ax.set_title(f"Raw boxplot: {col}")
plt.tight_layout(); plt.show()

## 3. Split first and fit an IQR Winsorizer

The IQR rule defines the lower limit as Q1 − 1.5×IQR and the upper limit as Q3 + 1.5×IQR. We calculate those limits from training rows only. `add_indicators=True` creates flags showing which observations were capped.

In [ ]:
X = data[features].copy()
X_train, X_test = train_test_split(X, test_size=0.30, random_state=0)
winsorizer = Winsorizer(capping_method="iqr", tail="both", fold=1.5, variables=features, add_indicators=True)
winsorizer.fit(X_train)
X_train_capped = winsorizer.transform(X_train)
X_test_capped = winsorizer.transform(X_test)
limits = pd.DataFrame({"lower_limit": pd.Series(winsorizer.left_tail_caps_), "upper_limit": pd.Series(winsorizer.right_tail_caps_)})
display(limits)
display(X_train_capped.head())

## 4. Compare raw and capped values

Capping changes only values beyond the learned limits. Row count stays unchanged. The indicator columns help audit how many observations were affected.

In [ ]:
summary = pd.DataFrame({"raw_min": X_train.min(), "capped_min": X_train_capped[features].min(), "raw_max": X_train.max(), "capped_max": X_train_capped[features].max()})
display(summary)
indicator_cols = [c for c in X_train_capped if c.endswith("_left") or c.endswith("_right")]
display(X_train_capped[indicator_cols].sum().rename("capped_rows").to_frame())
fig, axes = plt.subplots(2, len(features), figsize=(18, 8))
for i, col in enumerate(features):
    sns.boxplot(x=X_train[col], ax=axes[0, i], color="#4C78A8"); axes[0, i].set_title(f"Before: {col}")
    sns.boxplot(x=X_train_capped[col], ax=axes[1, i], color="#54A24B"); axes[1, i].set_title(f"After: {col}")
plt.tight_layout(); plt.show()

## 5. Compare statistical rules

The choice of `gaussian`, `iqr`, `mad`, or `quantiles` changes what is considered extreme. Gaussian limits use mean and standard deviation; IQR uses quartiles; MAD uses a robust median-based spread; quantiles cap fixed tails. Choose based on the distribution and domain, not because one method always wins.

In [ ]:
rows = []
for method, fold in [("gaussian", 3), ("iqr", 1.5), ("mad", 3.29), ("quantiles", 0.05)]:
    w = Winsorizer(capping_method=method, tail="both", fold=fold, variables=features)
    w.fit(X_train)
    for col in features: rows.append({"method": method, "feature": col, "lower": w.left_tail_caps_[col], "upper": w.right_tail_caps_[col]})
display(pd.DataFrame(rows).round(2))

## Summary and cautions

`Winsorizer` preserves rows but limits extreme values. Always fit it on training data, retain the fitted limits, and apply those same limits to future data. Capping can hide meaningful rare cases, so inspect flagged rows and compare model performance with and without the transformation. Missing values must be imputed first or handled with the transformer’s supported missing-value option.